## Baseline Evaluation: BGE-M3 on Spanish Legal Case Retrieval

**Objective**. This notebook evaluates the off-the-shelf BAAI/bge-m3
encoder on a dense legal case retrieval task in Spanish.


**Evaluation protocol**. We use the InformationRetrievalEvaluator from
sentence-transformers, which computes standard IR metrics (Acc@k, MRR@k, nDCG@k,
MAP@100) over a set of queries and a document corpus in dense vector space using
cosine similarity.


**Key design decision — corpus construction**.
In a realistic retrieval scenario, the model must identify the single correct passage
among a large pool of semantically similar but non-relevant documents. If the corpus
contained only the 60K eval passages (one per query), the task would be trivially easy:
every document in the corpus would be a correct answer to some query. To make the
evaluation meaningful, we augment the corpus with additional passages drawn from the
training split, which act as natural distractors. These passages come from the
same legal domain and share similar vocabulary and structure, but are not the
correct answer to any evaluation query. This forces the encoder to discriminate between
genuinely relevant passages and topically close but non-relevant ones — exactly the
challenge a retrieval system faces in production.





## 1. Environment Setup

Install the required libraries. We need `sentence-transformers >= 3.0.0` for the
`InformationRetrievalEvaluator`, `datasets` to load TripLegal-CL from Hugging Face,
and `accelerate` for GPU-optimized inference.


In [1]:
!pip install -q "sentence-transformers>=3.0.0" datasets accelerate


## 2. Imports and Device Configuration

In [2]:
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

Dispositivo: cuda


## 3. Hugging Face Authentication

The TripLegal-CL dataset may require authentication. Log in with your Hugging Face
token to access `wilfredomartel/TripLegal-CL`.

In [3]:
## Provide Hugging Face Token for Private Dataset Access
from huggingface_hub import notebook_login
notebook_login()

## 4. Load the Baseline Encoder

We load `BAAI/bge-m3` in its original, publicly available form
— **no legal domain adaptation has been applied**. This serves as the baseline
against which the fine-tuned version will be compared.

In [4]:
## loading the model
model_id = "BAAI/bge-m3"
model = SentenceTransformer(model_id, device=device)
print(f"The model has been loaded : {model_id}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

The model has been loaded : BAAI/bge-m3


### 5. Load and Prepare the Evaluation Data


We load the **last 200K contrastive instances** from TripLegal-CL
(indices 392,000–592,000) to avoid any overlap with data that could
be used for fine-tuning in subsequent experiments. From each instance,
we extract only the `query` and the first positive passage `pos[0]` —
the passage directly grounded in the source document.

The 200K instances are then split into:
- **Training split (70%):** ~140K instances — used here **only** as a
  source of distractor passages for the corpus, **not** for any model training.
- **Evaluation split (30%):** ~60K instances — provides the queries and
  their gold-standard passages.

  

### 5.1 Select the evaluation subset

In [5]:

dataset = load_dataset(
    "wilfredomartel/TripLegal-CL",
    split="train"
).select(range(392_000, 592_000)).shuffle(seed=42)

README.md:   0%|          | 0.00/428 [00:00<?, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00004-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/592382 [00:00<?, ? examples/s]

### 5.2 Extract query–passage pairs

From each contrastive instance, we extract only two fields:
- **`query`**: the natural-language legal question.
- **`passage`**: the first positive passage `pos[0]`, which is directly grounded
  in the source legal document.

All other fields (`neg`, `pos_score`, `neg_score`) are not needed for evaluation
and are removed.

In [6]:
def process_dataset(row):
    return {
        "query": row["query"],
        "passage": row["pos"][0]
    }

new_dataset = dataset.map(process_dataset, remove_columns=["pos", "neg", "pos_score", "neg_score"])
print(new_dataset.features)


Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

{'query': Value('string'), 'passage': Value('string')}


### 5.3 Split into train and eval partitions

The 200K instances are split 70/30:

| Partition | Size | Purpose in this notebook |
|-----------|------|--------------------------|
| **Train** (70%) | ~140K | Source of **distractor passages** for the corpus (not used for any model training) |
| **Eval** (30%) | ~60K | Provides the **queries** and their **gold-standard passages** |

> **Important:** The "train" partition here is **not** used for training. Its passages
> are borrowed solely to populate the retrieval corpus with realistic distractors.


In [7]:
split_datasets = new_dataset.train_test_split(test_size=0.3, seed=42)
train_dataset = split_datasets["train"]
eval_dataset = split_datasets["test"]

print(f" training_dataset size: {len(train_dataset)}")
print(f"Tamaño del eval_dataset_split: {len(eval_dataset)}")


 training_dataset size: 140000
Tamaño del eval_dataset_split: 60000


### 5.4 Inspect a sample instance

In [8]:

print("Q:", eval_dataset[0]["query"])
print("A:", eval_dataset[0]["passage"][:250], "...")

Q: ¿Por qué el IESS argumenta que la acción de protección debió ser rechazada por la Sala de lo Civil de la Corte Provincial de El Oro, citando la Ley Orgánica de Garantías Jurisdiccionales y Control Constitucional?
A: El Instituto Ecuatoriano de Seguridad Social (IESS) argumenta que la acción de protección debió ser rechazada por la Sala de lo Civil de la Corte Provincial de El Oro, basándose en el Art. 40 de la Ley Orgánica de Garantías Jurisdiccionales y Control ...


## 6. Build the Retrieval Benchmark: Queries, Corpus, and Relevance Judgments

In a dense retrieval evaluation, the model must find the **single correct passage**
for each query among a large pool of candidate documents. If the corpus only contained
the 60K eval passages (one per query), every document would be someone's correct answer
— making retrieval trivially easy. To create a **realistic and challenging** setting,
we augment the corpus with **domain-matched distractor passages** from the train partition.

The corpus layout is:

```
Index:    0 ──────────── 59,999 │ 60,000 ──────────── 119,999
          ◄── eval passages ──► │ ◄── train passages (distractors) ──►
          (gold answers)        │ (same legal domain, different cases)
```

This is standard practice in IR evaluation (cf. DPR — Karpukhin et al., 2020).

### 6.1 Build the query dictionary

Each evaluation query receives a unique integer ID `(0, 1, ..., 59,999)`.

In [9]:
queries = dict(enumerate(eval_dataset["query"]))
print(f"Total evaluation queries: {len(queries):,}")

Total evaluation queries: 60,000


### 6.2 Build the document corpus

In [10]:

# Source 1: Gold passages from the evaluation split (indices 0..59,999)
eval_passages  = list(eval_dataset["passage"])

# Source 2: Distractor passages from the training split (indices 60,000..119,999)
# These are legal passages from OTHER cases — NOT the answer to any eval query
train_passages = train_dataset["passage"][:60_000]

# Merge into a single corpus
corpus_passages = eval_passages + train_passages
corpus = dict(enumerate(corpus_passages))

print(f"Gold passages (eval split)   : {len(eval_passages):,}")
print(f"Distractors (train split)    : {len(train_passages):,}")
print(f"Total corpus size            : {len(corpus):,}")

Gold passages (eval split)   : 60,000
Distractors (train split)    : 60,000
Total corpus size            : 120,000


### 6.3 Build relevance judgments (qrels)

Since eval passages occupy corpus indices `0..59,999` and queries are also indexed
`0..59,999`, the relevance mapping is straightforward:

```
relevant_docs[i] = [i]   →   "the correct document for query i is corpus[i]"
```

Documents at indices ≥ 60,000 (the distractors) are **never** marked as relevant.

In [11]:
relevant_docs = {idx: [idx] for idx in queries}

# Sanity check
print(f"Query 0:        {queries[0][:150]}...")
print(f"Gold passage:   {corpus[0][:150]}...")
print(f"Relevant doc:   relevant_docs[0] = {relevant_docs[0]}")
print(f"\nDistractor example (corpus[60000]):")
print(f"  {corpus[60_000][:150]}...")
print(f"  → Different legal case, NOT relevant to query 0")


Query 0:        ¿Por qué el IESS argumenta que la acción de protección debió ser rechazada por la Sala de lo Civil de la Corte Provincial de El Oro, citando la Ley Or...
Gold passage:   El Instituto Ecuatoriano de Seguridad Social (IESS) argumenta que la acción de protección debió ser rechazada por la Sala de lo Civil de la Corte Prov...
Relevant doc:   relevant_docs[0] = [0]

Distractor example (corpus[60000]):
  El Inspector del Trabajo de Esmeraldas realizó una inspección focalizada a AGROCORONEL CIA. LTDA. con fundamento en una denuncia recibida el 15 de jun...
  → Different legal case, NOT relevant to query 0


## 7. Run the Information Retrieval Evaluation


We instantiate the `InformationRetrievalEvaluator` with the queries, corpus, and
relevance judgments defined above. The evaluator:

1. Encodes all 60K queries into dense vectors.
2. Encodes all 120K corpus documents into dense vectors.
3. Computes cosine similarity between each query and all corpus documents.
4. Ranks documents per query and computes the following metrics:

| Metric | Description |
|--------|-------------|
| **Acc@1** | Proportion of queries where the correct passage is ranked 1st |
| **Acc@10** | Proportion where the correct passage appears in the top 10 |
| **MRR@10** | Mean reciprocal rank of the correct passage within the top 10 |
| **nDCG@10** | Normalized discounted cumulative gain at rank 10 |
| **MAP@100** | Mean average precision up to 100 retrieved documents |



**Crear el InformationRetrievalEvaluator y ejecutar evaluación**

In [12]:
dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legalspanish-eval-60kq-120kd",
    show_progress_bar=True,
)

print("Evaluator created. Starting evaluation...")

results = dev_evaluator(model)

print("\nResults:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

Evaluator created. Starting evaluation...


Batches:   0%|          | 0/1875 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  33%|███▎      | 1/3 [08:18<16:36, 498.41s/it]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  67%|██████▋   | 2/3 [16:37<08:18, 498.90s/it]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 3/3 [19:58<00:00, 399.63s/it]



Results:
  legalspanish-eval-60kq-120kd_cosine_accuracy@1: 0.7460
  legalspanish-eval-60kq-120kd_cosine_accuracy@3: 0.8310
  legalspanish-eval-60kq-120kd_cosine_accuracy@5: 0.8556
  legalspanish-eval-60kq-120kd_cosine_accuracy@10: 0.8845
  legalspanish-eval-60kq-120kd_cosine_precision@1: 0.7460
  legalspanish-eval-60kq-120kd_cosine_precision@3: 0.2770
  legalspanish-eval-60kq-120kd_cosine_precision@5: 0.1711
  legalspanish-eval-60kq-120kd_cosine_precision@10: 0.0885
  legalspanish-eval-60kq-120kd_cosine_recall@1: 0.7460
  legalspanish-eval-60kq-120kd_cosine_recall@3: 0.8310
  legalspanish-eval-60kq-120kd_cosine_recall@5: 0.8556
  legalspanish-eval-60kq-120kd_cosine_recall@10: 0.8845
  legalspanish-eval-60kq-120kd_cosine_ndcg@10: 0.8161
  legalspanish-eval-60kq-120kd_cosine_mrr@10: 0.7941
  legalspanish-eval-60kq-120kd_cosine_map@100: 0.7969
